# IPI — data exploration starter

**Runs anywhere.** The root is discovered from the notebook's location, not hardcoded,
and every section checks whether its data is present before touching it. On a fresh
clone you get §1, §2 (small panel) and §4's log; §3 and the raw HTML need the full
data directory, which is not in git.

Storage is flat files — no database — in four tiers, each wanting a different tool:

| tier | where | size | in git? | tool |
|---|---|---|---|---|
| derived indices | `data/pilot/*.csv`, `docs/*.json` | KB | yes | `pd.read_csv` |
| price panels | `data/pilot/*-prices.csv` | 0.6–85 MB | small ones only | `pd.read_csv` |
| pipeline intermediates | `data/cdx-index/*.tsv` | 1.3–5.8 GB | **no** | **duckdb** |
| raw archived pages | `data/pilot/html*/` | 86 GB | **no** | `gzip.open`, one at a time |

A full clone is ~44 MB. The other 124 GB lives only on the collection machine.

In [ ]:
import os, sys, gzip, json, re
from pathlib import Path
import numpy as np, pandas as pd

def find_root(start=None):
    """Project root = nearest ancestor holding CLAUDE.md and data/. No hardcoded paths."""
    start = Path(start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / "CLAUDE.md").exists() and (p / "data").is_dir():
            return p
    raise RuntimeError(f"project root not found above {start}")

ROOT = find_root()
os.chdir(ROOT)                      # every data path below is repo-relative
sys.path.insert(0, str(ROOT / "code"))   # so `import gigfilter` works, as in the scripts

try:
    import duckdb
except ImportError:
    duckdb = None

print("root    ", ROOT)
print("pandas  ", pd.__version__)
print("duckdb  ", duckdb.__version__ if duckdb else "NOT INSTALLED — §3 will skip")

### Preflight — what data is actually on this machine

In [ ]:
def first_present(*paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

# price panels, largest first — the big two are gitignored, the small ones are tracked
# .gz first: the 88 MB CSVs are gitignored, the 8 MB gzips are committed and
# pd.read_csv reads them transparently. On a fresh clone this resolves to the
# same balanced panel the collection machine uses -- not a thin fallback.
PRICES = first_present("data/pilot/balanced-prices.csv", "data/pilot/balanced-prices.csv.gz",
                       "data/pilot/expanded-prices.csv", "data/pilot/expanded-prices.csv.gz",
                       "data/pilot/pilot-prices.csv")
GMI    = first_present("data/cdx-index/gig-month-index.tsv")
HTML   = first_present("data/pilot/html-balanced", "data/pilot/html-recent", "data/pilot/html")
NICHE  = first_present("data/pilot/niche-assignment.csv")

def mb(p):
    if p is None: return "-"
    if p.is_dir(): return "dir"
    return f"{p.stat().st_size/1e6:,.1f} MB"

for label, p in [("price panel", PRICES), ("cdx index", GMI), ("html store", HTML), ("niches", NICHE)]:
    print(f"{label:12s} {'OK ' if p else 'ABSENT':4s} {str(p or ''):45s} {mb(p)}")

## 1. Derived indices — tiny, git-tracked, always available

In [ ]:
ipi   = pd.read_csv("data/pilot/recent-ipi.csv")                     # quarter, ipi
panel = pd.read_csv("data/pilot/panel-ipi.csv")                      # long historical panel
cats  = pd.read_csv("data/pilot/recent-category-indices-geks.csv")   # quarter x 7 categories

display(ipi.tail())
display(cats.tail())

In [ ]:
# the numbers frozen into the paper and the website
paper = json.load(open("data/pilot/paper-numbers.json"))
site  = json.load(open("docs/data.json"))
print(sorted(paper)[:20])
print({k: site[k] for k in ("generated", "cadence", "base_period", "panel_gigs")})

## 2. The price panel

One row per (gig, archived snapshot) with the three Fiverr package tiers.
`file_path` points back to the exact archived HTML the price came from — that is
the lineage hook; use it whenever a number looks wrong.

In [ ]:
px = pd.read_csv(PRICES)
print(f"{PRICES}  ->  {px.shape[0]:,} rows x {px.shape[1]} cols, "
      f"{px.memory_usage(deep=True).sum()/1e6:.0f} MB resident")
px.head(3)

In [ ]:
(px.groupby("year")
   .agg(n=("price_basic", "size"),
        median_basic=("price_basic", "median"),
        sellers=("seller", "nunique"))
   .assign(median_basic=lambda d: d.median_basic.round(2)))

## 3. Big pipeline intermediates — duckdb, not pandas

`gig-month-index.tsv` is 1.3 GB and headerless; the classified / deduped page
tables are ~5.7 GB each. duckdb scans them out-of-core in seconds where
`pd.read_csv` would exhaust memory. **Gitignored — this section needs the
collection machine.**

Gotcha: the parameterised form `read_csv(..., columns=$cols)` with
`params={"cols": {...}}` is *silently ignored*; duckdb falls back to dialect
sniffing and then fails on a headerless TSV. Render the struct inline.

In [ ]:
if GMI is None or duckdb is None:
    print("SKIP — needs data/cdx-index/gig-month-index.tsv (1.3 GB, not in git) and duckdb")
else:
    GMI_SQL = (f"read_csv('{GMI}', delim='\\t', header=false, "
               "columns={'gig_id':'VARCHAR','month':'VARCHAR',"
               "'timestamp':'VARCHAR','category':'VARCHAR'})")

    display(duckdb.sql(f"""
        SELECT category, count(*) AS snapshots, count(DISTINCT gig_id) AS gigs
        FROM {GMI_SQL}
        GROUP BY 1 ORDER BY snapshots DESC
    """).df())

In [ ]:
if GMI is None or duckdb is None:
    print("SKIP — see above")
else:
    cov = duckdb.sql(f"""
        SELECT month, count(DISTINCT gig_id) AS gigs
        FROM {GMI_SQL}
        GROUP BY 1 ORDER BY 1
    """).df()
    print(f"{len(cov)} months, {cov.month.min()}-{cov.month.max()}")
    ax = cov.set_index("month").plot(figsize=(12, 3), legend=False,
                                     title="distinct gigs archived per month")
    ax.set_xlabel("")

## 4. Raw archived pages

`data/pilot/html-{recent,balanced}/<seller>/<YYYYMMDD>_<slug>.html.gz` — 86 GB.
Never glob the tree. Enter through the price panel's `file_path`, or through the
download logs.

In [ ]:
row  = px.iloc[0]
cand = [Path(row.file_path.replace(".html", ".html.gz")), Path(row.file_path)]
path = next((p for p in cand if p.exists()), None)

if path is None:
    print(f"SKIP — HTML store not on this machine (wanted {cand[0]})")
else:
    html = (gzip.open(path, "rt", errors="replace").read() if path.suffix == ".gz"
            else path.read_text(errors="replace"))
    print(path, f"{len(html):,} chars")
    print(re.search(r"<title>(.*?)</title>", html, re.S).group(1)[:120])

In [ ]:
# the download logs are the cheap index into the HTML store — and they ARE in git
log = pd.read_csv("data/pilot/recent-download-log.tsv", sep="\t")
print(log.shape, log.status.value_counts().to_dict())
log.head(3)

## 5. Niche assignment — the 2026-08 event-study work

In [ ]:
if NICHE is None:
    print("SKIP — niche-assignment.csv not committed yet")
else:
    arr = pd.read_csv("data/pilot/niche-arrival.csv")
    asg = pd.read_csv(NICHE)
    print(asg.shape, "usable:", int(asg.usable.sum()))
    display(arr.dropna(subset=["arrival_quarter"])
               .sort_values("n_listings", ascending=False).head(10))

## 6. Prices over time

From here on the notebook works on the **balanced panel** (`balanced-prices.csv`,
292k rows / 39.6k gigs after filters), built through `code/64-event-study-twfe.py`
so the filters are the ones steps 19/21 apply — `is_gig` on the handle, a known
category, `0 < price <= $10,000`, then the gig-quarter median.

Two warnings that govern every chart below.

**The series is truncated at 2024Q4 on purpose.** 2025-26 exists in the file but
is 5.6k and 650 rows against ~45k/year before it — the 403/PerimeterX cliff, not
a market that stopped trading. `plans/todo.md` carries an open blocker requiring
those points be suppressed or hard-banded before they reach any exhibit. Set
`END_Q` later if you want to look at them, but do not plot them as measurements.

**A raw median per quarter is not a price index.** It moves when the *mix* of
gigs changes, and this market's entrants price above its incumbents (step 57 §4),
so composition alone pushes the median up. §7 and §8 are the two ways of removing
that; the gap between this chart and those is the composition effect.

In [ ]:
import importlib.util
import matplotlib.pyplot as plt

def _load(name, rel):
    spec = importlib.util.spec_from_file_location(name, rel)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    return mod

esm  = _load("es",   "code/64-event-study-twfe.py")   # panel builder + event study
geks = _load("geks", "code/21-geks-index.py")         # GEKS-Jevons, imported not rewritten
tpd  = geks.tpd

CATEGORY_CSV = "data/pilot/balanced-gig-category.csv.gz"   # committed, 0.9 MB
START_Q, END_Q = "2020Q1", "2024Q4"
LAUNCH, BASE   = esm.CHATGPT_LAUNCH, esm.DEFAULT_BASE      # 2022Q4 / 2022Q3

gq = esm.build_panel(PRICES, CATEGORY_CSV)
print(f"{PRICES}")
print(f"  {gq.attrs['rows_in']:,} rows in -> {gq.attrs['rows_kept']:,} kept "
      f"-> {len(gq):,} gig-quarter cells, {gq.gig_id.nunique():,} gigs")
print(f"  quarters {gq.quarter.min()}..{gq.quarter.max()}, "
      f"median {gq.groupby('gig_id').size().median():.0f} quarters per gig")

# The balanced sample holds at most one capture per gig-quarter, so the median
# collapse is a near no-op here. It matters on the other panels; keep it.
gq.head(3)

In [ ]:
w = gq[(gq.qi >= esm.q_to_int(START_Q)) & (gq.qi <= esm.q_to_int(END_Q))]

overall = (w.groupby("quarter")
             .agg(median=("price_basic","median"), mean=("price_basic","mean"),
                  gigs=("gig_id","nunique"))
             .sort_index(key=lambda s: s.map(esm.q_to_int)))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                             gridspec_kw={"height_ratios":[3,1]})
a1.plot(overall.index, overall["median"], "o-", lw=2, label="median basic price")
a1.plot(overall.index, overall["mean"], "s--", lw=1.2, alpha=.7, label="mean")
a1.axvline(LAUNCH, color="crimson", ls=":", lw=2)
a1.annotate("ChatGPT\n2022-11-30", xy=(LAUNCH, a1.get_ylim()[1]),
            xytext=(4,-28), textcoords="offset points", color="crimson", fontsize=9)
a1.set_ylabel("USD"); a1.legend(frameon=False); a1.grid(alpha=.3)
a1.set_title("Listed basic price, all categories (unadjusted — composition not held fixed)")

a2.bar(overall.index, overall["gigs"], color="0.7")
a2.set_ylabel("gigs"); a2.grid(alpha=.3)
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

display(overall.assign(median=lambda d: d["median"].round(2),
                       mean=lambda d: d["mean"].round(2)))

In [ ]:
# by category — same caveat, composition is not held fixed
fig, ax = plt.subplots(figsize=(11, 5))
for c in tpd.CATS:
    s = (w[w.category == c].groupby("quarter").price_basic.median()
           .sort_index(key=lambda s: s.map(esm.q_to_int)))
    ax.plot(s.index, s.values, "o-", ms=3, lw=1.4, label=c)
ax.axvline(LAUNCH, color="crimson", ls=":", lw=2)
ax.set_ylabel("median basic price (USD)"); ax.grid(alpha=.3)
ax.legend(ncol=4, frameon=False, fontsize=9)
ax.set_title("Median listed price by category")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 7. GEKS-Jevons index

`geks_index()` is **imported from `code/21-geks-index.py`**, not reimplemented, so
what the notebook plots is the estimator the papers use. For every pair of quarters
it forms the direct bilateral Jevons comparison over gigs present in *both*,

$$\ln P(s,t) \;=\; \frac{1}{|G_{s,t}|}\sum_{i \in G_{s,t}} \bigl(\ln p_{i,t} - \ln p_{i,s}\bigr)$$

then makes those transitive by averaging every route through a link quarter $l$:

$$\ln P_{\text{GEKS}}(s,t) \;=\; \frac{1}{L}\sum_{l}\bigl[\ln P(s,l) + \ln P(l,t)\bigr]$$

The gig's price *level* cancels inside each bilateral difference, so there is no
fixed effect to estimate and no chain to drift along — this is a matched-model
index, and it answers §6's composition problem by construction. A pair needs
`MIN_MATCH = 3` matched gigs to count; `pair_density` reports the share of pairs
that clear it, and is the main way GEKS degrades on thin categories.

Bootstrap SEs resample **gigs** (200 draws in the pipeline; 100 here, ~70 s —
this is the slow cell). Base quarter is the first in the window, pinned to 100.

In [ ]:
import numpy as np

panel = esm.to_nested(w)                     # {category: {gig: {quarter: price}}}
rng   = np.random.default_rng(7)
N_BOOT = 100                                 # 0 to skip the bootstrap and run instantly

idx_by_cat, se_by_cat = {}, {}
for c in tpd.CATS:
    idx, se, diag = geks.geks_index(panel[c], rng=rng, n_boot=N_BOOT, window_start=START_Q)
    idx_by_cat[c], se_by_cat[c] = idx, se
    print(f"{c:12s} gigs={diag['gigs']:6,}  quarters={diag['quarters_out']:3d}  "
          f"pair_density={diag['pair_density']:.2f}")

geks_df = pd.DataFrame(idx_by_cat).sort_index(key=lambda s: s.map(esm.q_to_int))
geks_df.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for c in tpd.CATS:
    s  = geks_df[c].dropna()
    se = pd.Series(se_by_cat[c]).reindex(s.index)          # log-scale SE
    ax.plot(s.index, s.values, "o-", ms=3, lw=1.5, label=c)
    ax.fill_between(s.index, s.values*np.exp(-1.96*se), s.values*np.exp(1.96*se), alpha=.12)
ax.axhline(100, color="0.4", lw=.8)
ax.axvline(LAUNCH, color="crimson", ls=":", lw=2)
ax.annotate("ChatGPT", xy=(LAUNCH, ax.get_ylim()[1]), xytext=(4,-14),
            textcoords="offset points", color="crimson", fontsize=9)
ax.set_ylabel(f"GEKS-Jevons index ({START_Q} = 100)"); ax.grid(alpha=.3)
ax.legend(ncol=4, frameon=False, fontsize=9)
ax.set_title("GEKS-Jevons matched-model price index, by category (95% bootstrap bands)")
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## 8. Event study — the same gigs before and after ChatGPT

$$\ln p_{i,t} \;=\; \alpha_i \;+\; \sum_{q \neq \text{2022Q3}} \delta_q \,\mathbf{1}[t=q] \;+\; \varepsilon_{i,t}$$

$\alpha_i$ is a **gig** fixed effect, $\delta_q$ a **quarter** effect, and 2022Q3
— the last fully pre-launch quarter — is the omitted base, so every $\delta_q$
reads as a log change relative to it. ChatGPT was released **2022-11-30**, inside
2022Q4, so 2022Q4 is the first treated quarter.

The sample is restricted to gigs observed **at least once before and at least once
after** the cut. That is what makes this a within-gig comparison: the path cannot
move because different gigs entered or left, which is exactly the objection that
sinks a naive before/after here, since entrants price above incumbents.

Gig effects are absorbed by within-gig demeaning rather than estimated, so 16k
dummies never touch memory. **SEs are clustered on gig.**

> **What this identifies.** There is no control group — every gig meets ChatGPT on
> the same date — so $\delta_q$ is the common time path of price on a fixed panel,
> net of gig level. Calling any part of it a causal effect requires assuming price
> would have been *flat* without the launch. The next cell tests that assumption
> directly, and on this data it fails: the series is already rising steeply before
> ChatGPT exists. Designs that do have a control group are steps 50/53/58 and the
> niche design of step 61; all failed, and the diagnosis in `plans/todo.md` is this
> same pre-trend. Read §8 as the picture of that problem, not as a way around it.

In [ ]:
bal = esm.balanced_sample(gq, cut=LAUNCH, window=(START_Q, END_Q))
print(f"window {START_Q}..{END_Q}")
print(f"  gigs seen pre-{LAUNCH} : {bal.attrs['n_pre']:,}")
print(f"  gigs seen post        : {bal.attrs['n_post']:,}")
print(f"  BALANCED (both sides) : {bal.attrs['n_balanced']:,}   <- the estimation sample")

tab, diag = esm.event_study(bal, base=BASE)
print(f"\n{diag['obs']:,} obs, {diag['gigs']:,} gigs, {diag['quarters']} quarters, base {diag['base']}")
tab[["quarter","coef","se","ci_lo","ci_hi","pct"]].round(4)

In [ ]:
pt = esm.pretrend_test(tab, cut=LAUNCH)
print(f"pre-trend fitted on {pt['pre_quarters']} pre-launch quarters:")
print(f"  slope {pt['slope']:+.4f} log points/quarter  (t = {pt['t']:.1f})")
print(f"  gap at {tab.quarter.iloc[-1]}: {pt['gap_last']:+.4f} log points "
      f"({100*np.expm1(pt['gap_last']):+.1f}%) vs the extrapolated pre-trend")

t = pt["table"]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.axhspan(-99, 99, xmin=0, xmax=0, color="none")
ax.fill_between(t.quarter, t.ci_lo, t.ci_hi, alpha=.2, label="95% CI (gig-clustered)")
ax.plot(t.quarter, t.coef, "o-", lw=2, color="C0", label=r"$\delta_q$ (gig + quarter FE)")
ax.plot(t.quarter, t.pretrend, "--", lw=1.6, color="0.45",
        label="pre-launch linear trend, extrapolated")
ax.axvline(LAUNCH, color="crimson", ls=":", lw=2)
ax.axhline(0, color="0.4", lw=.8)
ax.annotate("ChatGPT 2022-11-30", xy=(LAUNCH, ax.get_ylim()[1]), xytext=(5,-16),
            textcoords="offset points", color="crimson", fontsize=9)
ax.set_ylabel(f"log price relative to {BASE}")
ax.set_title(f"Event study: {bal.attrs['n_balanced']:,} gigs observed both sides of ChatGPT")
ax.legend(frameon=False, loc="upper left"); ax.grid(alpha=.3)
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

In [ ]:
# Same design run inside each category — does any one of them break at the launch?
rows = []
for c in tpd.CATS:
    sub = esm.balanced_sample(gq[gq.category == c], cut=LAUNCH, window=(START_Q, END_Q))
    if sub.attrs["n_balanced"] < 200:
        rows.append({"category": c, "gigs": sub.attrs["n_balanced"], "note": "too thin"}); continue
    ct, _ = esm.event_study(sub, base=BASE)
    cpt   = esm.pretrend_test(ct, cut=LAUNCH)
    rows.append({"category": c, "gigs": sub.attrs["n_balanced"],
                 "pre_slope": round(cpt["slope"], 4), "pre_t": round(cpt["t"], 1),
                 "post_gap": round(cpt["gap_last"], 4),
                 "cum_2024Q4": round(float(ct.coef.iloc[-1]), 4)})
pd.DataFrame(rows)

### Reading the result

Three things to take from §8, in the order they should change your mind.

1. **The pre-trend is the headline.** Prices rise ~4.7 log points *per quarter*
   through 2020Q1–2022Q3, and a straight line fits those 11 quarters to within
   ±0.02. Whatever is moving this market was moving it hard for two years before
   ChatGPT shipped. Any claim of the form "prices did X after the launch" has to
   beat that line, not zero.

2. **There is no break at the launch.** 2022Q4 and 2023Q1 sit on the pre-trend;
   nothing steps, in either direction, at the date. This reproduces the papers'
   central null on a different estimator than the ones they use.

3. **The post-period deceleration is real but weakly identified.** The series keeps
   rising and ends 2024Q4 about 0.21 log points (≈19%) *below* where the pre-trend
   would have put it. That is a large number and it is the most interesting thing
   on the chart — but it rests on extrapolating a linear trend eight quarters past
   its last data point, and no price series grows exponentially forever. Mean
   reversion produces the same picture with no treatment at all, and step 49 shows
   that exact failure mode on this data. Treat it as a description of the shape,
   not an effect, until it survives a control group.

The honest one-line summary: **prices were rising steeply before ChatGPT, did not
break at it, and rose more slowly afterwards for reasons this design cannot
attribute.**